# Hasil eksekusi Colab: 03a_knn_manual

Dijalankan di VM Colab GPU T4 (sesi `d920cd`). Sel di bawah adalah kode notebook asli beserta outputnya.


In [ ]:
import pandas as pd
import numpy as np
import math
import random

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

Ukuran dataset bersih: (5109, 7)
BMI kosong: 0
5 baris pertama:


,gender,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,Male,67.0,0,1,228.69,36.60000,1
1,Female,61.0,0,0,202.21,31.84859,1
2,Male,80.0,0,1,105.92,32.50000,1
3,Female,49.0,0,0,171.23,34.40000,1
4,Female,79.0,1,0,174.12,24.00000,1


In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

Train: 4088 baris
Test:  1021 baris
Proporsi stroke di train: 0.046
Proporsi stroke di test:  0.060


In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

In [ ]:
def euclidean_distance(a, b):
    """Jarak Euclidean antara dua vektor."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def knn_predict(X_train, y_train, X_test, k=5):
    """KNN: prediksi kelas untuk tiap titik uji."""
    y_pred = []

    for titik_uji in X_test:
        jarak = []
        for i, titik_latih in enumerate(X_train):
            d = euclidean_distance(titik_uji, titik_latih)
            jarak.append((d, y_train[i]))

        jarak.sort(key=lambda x: x[0])
        tetangga = jarak[:k]

        voting = 0
        for _, label in tetangga:
            voting += label

        pred = 1 if voting > k / 2 else 0
        y_pred.append(pred)

    return np.array(y_pred)


print("Menjalankan KNN (k=5)...")
y_pred_knn = knn_predict(X_train, y_train, X_test, k=5)
print("Selesai.")

Menjalankan KNN (k=5)...


Selesai.


In [ ]:
metrik_knn = hitung_metrik(y_test, y_pred_knn)

print("=" * 50)
print("HASIL KNN (k=5)")
print("=" * 50)
print(f"  Akurasi:  {metrik_knn['akurasi']}")
print(f"  Presisi:  {metrik_knn['presisi']}")
print(f"  Recall:   {metrik_knn['recall']}")
print(f"  F1-score: {metrik_knn['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_knn['tp']}  FP={metrik_knn['fp']}")
print(f"  FN={metrik_knn['fn']}  TN={metrik_knn['tn']}")

HASIL KNN (k=5)
  Akurasi:  0.9285
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0

Confusion Matrix:
  TP=0  FP=12
  FN=61  TN=948


In [ ]:
print(f"{'K':>3} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 40)

for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    pred = knn_predict(X_train, y_train, X_test, k=k)
    m = hitung_metrik(y_test, pred)
    print(f"{k:>3} {m['akurasi']:>8} {m['presisi']:>8} {m['recall']:>8} {m['f1']:>8}")

  K  Akurasi  Presisi   Recall       F1
----------------------------------------


  1   0.9197   0.2308   0.1475     0.18


  3   0.9285    0.125   0.0328   0.0519


  5   0.9285      0.0      0.0        0


  7   0.9334      0.0      0.0        0


  9   0.9383      0.0      0.0        0


 11   0.9393      0.0      0.0        0


 15   0.9403        0      0.0        0


 21   0.9403        0      0.0        0


In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

Sebaran skor pada data uji:
0    252
1    462
2    160
3     89
4     45
5     13


In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

,Aturan,akurasi,presisi,recall,f1,ditandai
0,skor >= 1,0.2987,0.0741,0.9344,0.1373,769
1,skor >= 2,0.7218,0.1368,0.6885,0.2283,307
2,skor >= 3,0.8570,0.2109,0.5082,0.2981,147
3,skor >= 4,0.9167,0.2931,0.2787,0.2857,58
